# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from os.path import join, exists

# Import Functions
sys.path.append("../../")

from src.configs.default_configs import fn_ue_perf
from src.configs.octmnist_config import data_name, target_col, num_classes
from src.file_manager.filepath import FilePath
from src.file_manager.load_save_df import load_all_pred_dfs
from model_ue_dict import ModelClass_dict, ue_dict
from src.evaluation.evaluate import evaluate_ue
from src.evaluation.ue_ood_eval import evaluate_all_ue_ood
from src.evaluation.perf_eval import get_prediction_performance_table, display_pred_perf

from cur_seed import seed
# seed = 2024

fp = FilePath(data_name=data_name, seed=seed)

# Load Predictions

In [ ]:
pred_df = load_all_pred_dfs(fp, ModelClass_dict=ModelClass_dict)

# Prediction Performance

In [ ]:
pred_perf_df = get_prediction_performance_table(
    pred_df, ue_dict, target_col, num_classes, split_col="split_perf", test_label="Test-OCTMNIST")
display_pred_perf(pred_perf_df)
pred_perf_df.to_csv(join(fp.get_parent_folder(fn_ue_perf), "pred_perf.csv"))

In [ ]:
pred_perf_df = get_prediction_performance_table(
    pred_df, ue_dict, target_col, num_classes, split_col="split_perf", test_label="Test-OCTDL")
display_pred_perf(pred_perf_df)
pred_perf_df.to_csv(join(fp.get_parent_folder(fn_ue_perf), "pred_perf_out.csv"))

# Evaluate UE

In [ ]:
pred_df = load_all_pred_dfs(fp, ModelClass_dict=ModelClass_dict)
evaluate_ue(pred_df=pred_df, ue_dict=ue_dict, fp=fp)

# OOD Detection

In [ ]:
pred_df = load_all_pred_dfs(fp, ModelClass_dict=ModelClass_dict)
pred_df_ood_labels = {
    "OCTDL IN": "ood_in_octdl", "OCTDL OUT": "ood_out_octdl", "ChestMNIST": "ood_chestmnist"}
pred_df_ood_dict = {}
for name, label in pred_df_ood_labels.items():
    pred_df_ood_dict[name] = load_all_pred_dfs(
        fp, ModelClass_dict=ModelClass_dict, pred_optional_label=label, index="single")
ood_eval_df = evaluate_all_ue_ood(
    pred_df=pred_df, pred_df_ood_dict=pred_df_ood_dict, ue_dict=ue_dict, 
    split_col="split_perf", test_label="Test-OCTMNIST", fp=fp
)